In [ ]:
# Produce Processing-structure-performance knowledge graphs
# and the associated predictive models
# Larry Lueer, 01.08.2023

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.ml import Mrmr_gpr
mg = Mrmr_gpr()

In [ ]:
# Some settings:
missing = -99999 # value which means "missing data"

In [ ]:
# Load a Pandas DatFrame.
df = pd.read_excel('Data/Area H TC TimeLOG Model.xlsx')

In [ ]:
# To make life simpler for the predictive model, remove all non-numeric columns

df1n = df.select_dtypes(include='number', exclude=None)
print(f'after removing non-numeric columns: shape {df1n.shape}')

# remove outliers, if desired (handle with care!)
# fe.outliers_to_missing(df1n, missing=missing, z_score=5, method='madm')

# finally, replace all "missing" values with np.nan
df1n.replace(to_replace=missing, value= np.nan, inplace=True)
colnan = np.sum([np.any(np.isnan(df1n.iloc[:,ii].to_numpy())) for ii in range(len(df1n.columns))])
rownan = np.sum([np.any(np.isnan(df1n.iloc[ii,:].to_numpy())) for ii in range(len(df1n))])
print(f'Number of columns with nans: {colnan}')
print(f'Number of rows with nans: {rownan}')

# At this point, the Pandas DataFrame is ready for building predictive models

In [ ]:
# Familiarize yourself with the dataset. Very important!
# Don't rely on AI! Your insight is key!
# For example, from our Physics knowledge we expect the acceptor exciton energy
# to influence the open circuit voltage

#fe = Features() # a handy class for things like Pearson correlation etc.
#predictor1 = 'features_A.1.c_UV' # feature for the exciton energy (in eV)
#predictor2 = 'xlsx_Voc (V)' # open circuit voltage (in V)
#fig,ax = fe.two_predictors(df,predictor1,predictor2)  
#fig.set_size_inches(4, 3, forward=True)

In [ ]:
################### 240502: run this cell instead of the one above for error testing!!! ###################

# in the "hierachical mrmr", you define the hierarchy of features
# select the features which describe process conditions (P), microstructure (S) and terget objectives (T)
#features_P = [cc for cc in df1n if cc.startswith('plan')] # features for the process conditions
features_P = []
features_S = [ 'B', 'C', 'D', 'E', 'F', 'G', 'H' , 'I' , 'J',  'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y'] # features for the microstructure
targets = ['Target'] # the targets for the predictive model

excludes = [[] for _ in targets]

In [ ]:
# define settings & start h_mrmr_gpr
use_gpr = True
add_P = False # add process conditions?
n_add_layers = 0 # how many additional layers should be rendered AFTER the best predictors for the targets?
regressor_params = {'length_scale_lims':(1,50), 'length_scales_start':1}
proxy_params = {'select_method':'dR2', 'missing':-99999, 'min_rel_overlap':0.3, 'normalize_y':True, 'n_shuffle':20} 


# we need to make sure that there is no single nan in the table that the upstream mrmr gets
# therefore we use Features.handle_missing() with the "keep datasets" option. It will remove entire predictors
# if they contain a single nan.

print(f'incoming dataset: shape {df1n.shape}')
df1nn, pred_out, targ_out = mg.handle_missing(df1n, features_S, targets, predictors_whitelist=[], predictors_blacklist=[],
                        targets_whitelist=[], targets_blacklist=[], missing=-99999, handle_missing='keep_datasets', min_var=2)
print(f'after removing columns with nans: shape {df1nn.shape}')

In [ ]:
# now the dataset is ready for the h-mrmr algorithm of the Gpr_ucb class
#gu = Gpr_ucb() # initialize the class
mg = Mrmr_gpr()

mg.regressor = 'gpr' # 'gpr', 'rf' or 'gbr'; you can also define a custom regressor

gax = mg.h_mrmr(df1nn, targ_out, excludes, whitelist_S=pred_out, whitelist_P=features_P,
                max_pred_mrmr = 25, max_pred_gpr =3, n_add_layers = 0,
                add_P = add_P, graph = None, use_gpr = use_gpr, regressor_params = regressor_params,
                proxy_params = proxy_params, whitelist_exact_match = True, required = [])

mg.graph_draw_mplx(G=gax,gencolors={'P':'black','S':'teal','T':'darkmagenta'}) # finally, draw the resulting knowledge graph

In [ ]:
# look at selected predictive models
targets = ['Target'] 
features = ['P','I','D']

missing=-1
length_scales_start =1# was 1
length_scale_lims = (1,10000) # use this if predictors will be standardized
mg = Mrmr_gpr()
mg.regressor = 'gpr' # 'gpr', 'rf' or 'gbr' 
select_method = 'dR2'
min_rel_overlap = 0.6 # WAS: 0.85
missing =-1
normalize_y = True # important for kr, knr

r = mg.fit(df1nn,features,targets[0],exclude=missing,min_var=1,min_rel_overlap=0,min_grow=0,min_pts=10, n_shuffle=20,
            length_scales_start=length_scales_start, length_scale_lims=length_scale_lims,normalize_y=normalize_y)
fig1 = mg.plot_result_plot(fn='result_plot.xlsx',n_jacs=-1) # plot predicted against ground truth for each data point

if len(features)>1:
    #fig2=mg.plot_jacs_1d(deriv=1,p_names=features) # plot first derivatives of obj funcs (only for more than 2 predictors!)
    fig3=mg.plot_predictor_pairs() # new, nice 3D
fig, axs = mg.plot_partial_dependence_1D()

In [ ]:
# 保存高清图片
fig1.savefig('0.716_result_plot_highres.png', dpi=600, bbox_inches='tight')

if 'fig3' in locals() and fig3 is not None:
    if isinstance(fig3, tuple):
        fig3[0].savefig('0.716_predictor_pairs_highres.png', dpi=600, bbox_inches='tight')
    else:
        fig3.savefig('0.716_predictor_pairs_highres.png', dpi=600, bbox_inches='tight')

if 'fig' in locals() and fig is not None:
    if isinstance(fig, tuple):
        fig[0].savefig('3_partial_dependence_1D_highres.png', dpi=600, bbox_inches='tight')
    else:
        fig.savefig('0.716_partial_dependence_1D_highres.png', dpi=600, bbox_inches='tight')